# 8.2 · 随机森林深入 / Random Forest Deep Dive

> **课程定位 / Where this fits**
> 第 2 课，**Part 8 · 集成学习**。
> Lesson 2, **Part 8 · Ensemble Learning**.
>
> 8.1 的 bagging 公式留了一个尾巴：方差下限 $\rho\sigma^2$ 卡在**树之间的相关性 $\rho$**。随机森林的精髓就是在 bagging 上**再加一层随机——每次分裂只看随机的部分特征**，强行让树彼此不同(降 $\rho$)，把方差压得更低。5.7 已介绍过 RF，这一课**深入**：实测 max_features 如何降相关、ExtraTrees 的极端随机、以及特征重要性的偏差与正确做法。
> Bagging (8.1) left a loose end: the variance floor $\rho\sigma^2$ is set by the **correlation $\rho$ between trees**. Random Forest's essence is adding **one more layer of randomness — each split only considers a random subset of features** — forcing trees to differ (lowering $\rho$) and crushing variance further. RF was introduced in 5.7; this lesson goes **deep**: measuring how max_features lowers correlation, ExtraTrees' extreme randomness, and importance bias with the right fix.
>
> 💼 **实战/面试视角**："随机森林的'随机'在哪 / 为什么特征随机 / max_features 怎么调 / 重要性的坑" 高频。
> 💼 **Practical/interview angle:** "where's the randomness / why feature randomness / tuning max_features / importance pitfalls" are common.

> 📐 **符号约定 / Notation**
> - $m$ (`max_features`) —— 每次分裂随机考虑的特征数 / features per split
> - $\rho$ —— 树之间预测的相关系数 / correlation between trees

> 💡 **面试相关 / Interview-relevant**
> - "随机森林的两处随机 / 为什么特征随机能降方差"（出镜率 ★★★★★）
> - "max_features 怎么影响偏差方差"（★★★★）
> - "ExtraTrees 和 RF 的区别"（★★★）
> - "不纯度重要性的偏差 / 用置换重要性"（★★★★★）
> - "RF 会过拟合吗 / 树越多越好吗"（★★★★）

---

## 学习目标 / Learning Objectives

1. 实测 **max_features 越小 → 树越去相关（ρ 越低）**。
   Measure that smaller max_features → more de-correlated trees (lower ρ).
2. 理解 max_features 的偏差方差权衡并调参。
   Understand max_features's bias-variance trade-off and tune it.
3. 了解 **ExtraTrees**（极端随机树）的额外随机。
   Know ExtraTrees' extra randomness.
4. 看清**不纯度重要性的偏差**，用置换重要性纠正。
   See the bias of impurity importance and fix with permutation importance.

## 目录 / TOC
1. [先建直觉：从 bagging 到 RF ⭐](#1)
2. [🚢 数据 + 实测特征随机降相关 ⭐](#2)
3. [max_features 调参 ⭐](#3)
4. [ExtraTrees：更随机 ⭐](#4)
5. [特征重要性的偏差 ⭐](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 先建直觉：从 bagging 到 RF ⭐ / Intuition: From Bagging to RF

回顾 8.1 的降方差公式 $\text{Var}=\rho\sigma^2+\frac{1-\rho}{B}\sigma^2$。加树（增大 $B$）只能消掉第二项，第一项 $\rho\sigma^2$ 卡死在那里——**只要树彼此相似（$\rho$ 大），方差就降不下去**。
Recall 8.1's formula $\text{Var}=\rho\sigma^2+\frac{1-\rho}{B}\sigma^2$. Adding trees (larger $B$) only kills the second term; the first $\rho\sigma^2$ is stuck — **as long as trees are similar (large $\rho$), variance won't drop further**.

普通 bagging 的树为什么相似？因为**如果有一个超强特征**（如 Titanic 的 sex），每棵树都会**先按它分裂**，于是长得很像。随机森林的解法绝妙：**每个分裂点只从随机的 $m$ 个特征里挑**（默认分类 $m=\sqrt{d}$，回归 $m=d/3$）。这样有时强特征根本不在候选里，逼不同的树去用不同的特征 → **树被去相关 → $\rho$ 降 → 方差降得更多**。
Why are plain-bagging trees similar? Because **with one super-strong feature** (e.g. Titanic's sex), every tree **splits on it first** and looks alike. RF's fix is elegant: **each split considers only $m$ random features** (default $m=\sqrt{d}$ for classification, $d/3$ for regression). Sometimes the strong feature isn't even a candidate, forcing trees to use different features → **de-correlated → lower $\rho$ → more variance reduction**.

所以 **RF = bagging + 特征随机**。下面**实测**这第二层随机确实降低了树间相关性。
So **RF = bagging + feature randomness**. Below we **measure** that this second randomness really lowers tree correlation.


<a id="2"></a>
## 2. 数据 + 实测特征随机降相关 ⭐ / Measuring De-correlation

用 **Titanic**。直接验证核心机制：对比"只 bagging（每分裂用全部特征）"和"RF（每分裂用 √d 个特征）"——后者的**树间预测相关性应明显更低**。我们用每对树在测试集上预测的平均相关系数来量化。
Using **Titanic**. We directly verify the core mechanism: compare "bagging only (all features per split)" vs "RF (√d features per split)" — the latter's **inter-tree prediction correlation should be clearly lower**. We quantify it via the average pairwise correlation of trees' test predictions.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
sns.set_theme(style="whitegrid")

df = sns.load_dataset("titanic")
feat = ["pclass", "sex", "age", "sibsp", "parch", "fare"]
d = df[feat + ["survived"]].copy()
d["age"] = d["age"].fillna(d["age"].median()); d["fare"] = d["fare"].fillna(d["fare"].median())
d["sex"] = (d["sex"] == "male").astype(int)
X, y = d[feat].values, d["survived"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

def avg_tree_correlation(rf, X):
    # 取每棵树对测试集的预测, 算两两相关系数的平均 / mean pairwise correlation of tree predictions
    P = np.array([t.predict(X) for t in rf.estimators_])     # (n_trees, n_test)
    C = np.corrcoef(P)                                        # 树×树 相关矩阵
    return C[np.triu_indices_from(C, k=1)].mean()            # 取上三角(不含对角)平均

# max_features=None → 每分裂用全部特征 = 纯 bagging; "sqrt" → RF / bagging vs RF
rf_bag = RandomForestClassifier(n_estimators=100, max_features=None, random_state=0).fit(X_tr, y_tr)
rf_rf  = RandomForestClassifier(n_estimators=100, max_features="sqrt", random_state=0).fit(X_tr, y_tr)
print(f"纯 bagging(每分裂用全部特征) 树间相关性 ρ ≈ {avg_tree_correlation(rf_bag, X_te):.3f}")
print(f"随机森林(每分裂用 √d 个特征)  树间相关性 ρ ≈ {avg_tree_correlation(rf_rf, X_te):.3f}")
print("→ 特征随机让树间相关性明显降低 → 据 8.1 公式, ρ↓ 则方差↓ → RF 比纯 bagging 更稳")


<a id="3"></a>
## 3. max_features 调参 ⭐ / Tuning max_features

`max_features` 是 RF 最重要的超参，它直接控制去相关程度，带来一个权衡：
`max_features` is RF's most important hyperparameter, directly controlling de-correlation, with a trade-off:
- **太小**：树高度去相关（$\rho$ 很低，方差很低），但每棵树**能用的特征太少 → 单棵树太弱（偏差升高）**。
  **Too small:** highly de-correlated (low $\rho$, low variance), but each tree has too few features → **too weak (higher bias)**.
- **太大**（=全部特征）：退化成纯 bagging，树相似（$\rho$ 高，方差降不够）。
  **Too large** (=all features): degenerates to plain bagging, similar trees (high $\rho$, insufficient variance reduction).
- 默认 $\sqrt{d}$（分类）/ $d/3$（回归）通常是好折中，但值得用 CV 验证。
  Defaults $\sqrt{d}$ (classification) / $d/3$ (regression) are usually a good balance, but worth verifying with CV.


In [ ]:
mfs = [1, 2, 3, 4, 5, 6]                                  # 每分裂考虑的特征数(共 6 个特征)
cv_scores = [cross_val_score(RandomForestClassifier(200, max_features=m, random_state=0), X, y, cv=5).mean()
             for m in mfs]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(mfs, cv_scores, "o-")
ax.axvline(np.sqrt(6), color="r", ls="--", label="√d 默认 ≈ 2.45")
best = mfs[int(np.argmax(cv_scores))]
ax.scatter([best], [max(cv_scores)], c="g", s=120, zorder=5, label=f"最优 {best}")
ax.set_xlabel("max_features (每分裂的特征数)"); ax.set_ylabel("CV 准确率"); ax.legend()
ax.set_title("max_features: 太小单树弱(偏差↑), 太大树相似(方差↓不够); 中等最优")
plt.tight_layout(); plt.show()
print(f"各 max_features 的 CV: {dict(zip(mfs, np.round(cv_scores,3)))}")
print(f"最优 max_features = {best} (√d≈2.45 通常接近最优); 是去相关 vs 单树强度的权衡")


<a id="4"></a>
## 4. ExtraTrees：更随机 ⭐ / Extra-Trees: Even More Random

**Extra-Trees（极端随机树）** 比 RF 多一层随机：RF 在候选特征里**找最优分裂点**，而 ExtraTrees **连分裂点也随机选**（不搜最优阈值，直接随机取）。这让树更去相关、**方差更低、训练更快**（省去找最优阈值的搜索），代价是**单棵树偏差略升**。很多时候它和 RF 不相上下、甚至更好，是值得一试的替代。
**Extra-Trees (Extremely Randomized Trees)** adds another layer of randomness: where RF **searches for the best split threshold** among candidate features, ExtraTrees **also picks the threshold at random** (no optimal-threshold search). This makes trees more de-correlated, **lower variance, faster training** (skips the threshold search), at the cost of **slightly higher per-tree bias**. Often on par with or better than RF — a worthwhile alternative to try.


In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
import time

for name, model in [("随机森林 RandomForest", RandomForestClassifier(n_estimators=200, random_state=0)),
                    ("极端随机树 ExtraTrees", ExtraTreesClassifier(n_estimators=200, random_state=0))]:
    t = time.perf_counter()
    acc = cross_val_score(model, X, y, cv=5).mean()
    print(f"{name:<22} CV 准确率 {acc:.3f}, 训练耗时 {time.perf_counter()-t:.2f}s")
print("\nExtraTrees 连分裂阈值也随机 → 更去相关、训练更快(不搜最优阈值); 单树偏差略升")
print("实战常和 RF 一起试, 取更优的那个")


<a id="5"></a>
## 5. 特征重要性的偏差 ⭐ / The Bias of Feature Importance

RF 自带的 `feature_importances_`（基于不纯度下降）**有一个重要偏差**（面试高频）：它**偏向高基数/连续特征**——因为连续特征能切出更多分裂点，更容易"碰巧"降低不纯度。下面用一个**和目标完全无关的随机连续特征**戳穿它：不纯度重要性会给这个噪声特征不小的分数。
RF's built-in `feature_importances_` (impurity-based) has an important bias (frequently asked): it **favors high-cardinality/continuous features** — continuous features offer more split points and more easily "happen to" reduce impurity. We expose it below with a **random continuous feature unrelated to the target**: impurity importance assigns it a non-trivial score.

正确做法是**置换重要性(permutation importance, 7.8)**：打乱某列看测试性能掉多少。它**模型无关、在测试集算、没有这个偏差**，会正确地给噪声特征几乎 0 分。
The fix is **permutation importance (7.8)**: shuffle a column and see how much test performance drops. It's **model-agnostic, computed on test, and free of this bias**, correctly giving the noise feature near-zero.


In [ ]:
from sklearn.inspection import permutation_importance

# 加一个与目标完全无关的随机连续特征 / a random continuous feature unrelated to y
rng = np.random.default_rng(0)
X_aug = np.c_[X, rng.normal(size=len(X))]
names = feat + ["随机噪声(连续) random_noise"]
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(X_aug, y, test_size=0.3, stratify=y, random_state=0)
rf = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xa_tr, ya_tr)

imp_impurity = pd.Series(rf.feature_importances_, index=names)
perm = permutation_importance(rf, Xa_te, ya_te, n_repeats=30, random_state=0)
imp_perm = pd.Series(perm.importances_mean, index=names)
cmp = pd.DataFrame({"不纯度重要性": imp_impurity, "置换重要性": imp_perm}).sort_values("不纯度重要性", ascending=False)
print(cmp.round(3).to_string())
print(f"\n关键对比 '随机噪声(连续)':")
print(f"  不纯度重要性 = {imp_impurity['随机噪声(连续) random_noise']:.3f}  ← 非0! (连续特征被高估)")
print(f"  置换重要性   = {imp_perm['随机噪声(连续) random_noise']:.3f}  ← ≈0 (正确识别为无用)")
print("→ 不纯度重要性偏向连续/高基数特征; 实战汇报特征重要性请用置换重要性(7.8)")


<a id="6"></a>
## 6. 小结 / Summary

```
RF = bagging + 特征随机(每分裂只看 √d 个特征) → 降树间相关 ρ → 据 8.1 公式方差降更多
实测: RF 的树间相关性明显低于纯 bagging(全特征分裂)
max_features: 太小→单树弱(偏差↑); 太大→树相似(方差降不够); √d(分类)/d/3(回归) 默认
ExtraTrees: 连分裂阈值也随机 → 更去相关+训练更快, 单树偏差略升; 值得一试
特征重要性: 不纯度重要性偏向连续/高基数特征(给无关噪声非0分) → 用置换重要性(7.8)
RF 树越多越好不过拟合(5.7/8.1); 不需缩放(继承自树)
```

### 💡 面试速查 / Interview cheat-sheet
1. **RF = bagging + 特征随机**; 特征随机的目的是**降树间相关 ρ**(降方差)。
   RF = bagging + feature randomness; the point is to lower inter-tree correlation ρ (variance).
2. **max_features 小→去相关强但单树弱**, √d/d3 是默认折中。
   Smaller max_features → more de-correlation but weaker trees; √d/d3 is the default balance.
3. **ExtraTrees 连阈值也随机** → 更随机+更快, 偏差略升。
   ExtraTrees randomizes the threshold too → more random/faster, slightly higher bias.
4. **不纯度重要性偏向连续/高基数特征** → 用置换重要性。
   Impurity importance favors continuous/high-cardinality features → use permutation importance.
5. **RF 树越多越好不过拟合**, 不需缩放。
   More trees never overfit RF; no scaling needed.

### 下一节 / Next
**8.3 AdaBoost**——从 bagging 切到 boosting。AdaBoost 串行训练弱学习器, 每轮**加大上一轮错分样本的权重**, 让后续学习器专攻难例。
**8.3 AdaBoost** — from bagging to boosting. AdaBoost trains weak learners sequentially, **up-weighting the previous round's misclassified samples** so later learners focus on the hard cases.
